In [62]:
import pandas as pd

# Reading in the Raw Data

In [63]:
df = pd.read_csv("../datasets/raw/eia_seds_data.csv")
df.head()

,period,seriesId,seriesDescription,stateId,stateDescription,value,unit
0,2023,CLEIV,Coal expenditures in the electric power sector,AK,Alaska,49.4,Million dollars
1,2023,CLEIV,Coal expenditures in the electric power sector,AL,Alabama,605.9,Million dollars
2,2023,CLEIV,Coal expenditures in the electric power sector,AR,Arkansas,387.6,Million dollars
3,2023,CLEIV,Coal expenditures in the electric power sector,AZ,Arizona,402.9,Million dollars
4,2023,CLEIV,Coal expenditures in the electric power sector,CA,California,0.0,Million dollars


The way this file is organized, each stat we have has its own row per state per year.

Here we can see that every expenditure value is measured in Millions of Dollars. Every Emissions value is measured in Millions of Metric tons of CO2

In [64]:
unique_units = df.groupby(["seriesId", "seriesDescription"])["unit"].unique()
unique_units

seriesId  seriesDescription                                                                           
CLEIV     Coal expenditures in the electric power sector                                                            [Million dollars]
CLTCE     Coal CO2 emissions for all sectors                                                              [Million metric tons CO2  ]
FFTCE     Fossil fuel CO2 emissions for all sectors                                                         [Million metric tons CO2]
NGEIV     Natural gas expenditures in the electric power sector (including supplemental gaseous fuels)              [Million dollars]
NNTCE     Natural gas, excluding supplemental gaseous fuels, CO2 emissions for all sectors                  [Million metric tons CO2]
NUEGV     Nuclear fuel expenditures in the electric power sector                                                    [Million dollars]
PAEIV     All petroleum products total expenditures in the electric power sector             

# Cleaning the Data

We want there to just be a single row per year per state. So we need to pivot it. 

We can get rid of the state description because we have the stateid

We can get rid of the series description because we can reference the table above to see the description that corresponds to id

We can get rid of units because all emissions and expenditure data have the same units, which we can reference from above.

In [65]:
df.drop(["seriesDescription", "stateDescription", "unit"], axis=1, inplace=True)
df = df.pivot_table(
    index=["stateId", "period"],
    columns="seriesId",
    values="value"
).reset_index()
# helps with merging later
df.rename(columns={"stateId": "stateid"}, inplace=True)
df

seriesId,stateid,period,CLEIV,CLTCE,FFTCE,NGEIV,NNTCE,NUEGV,PAEIV,PMTCE
0,AK,2016,29.7,1.6,33.4,187.7,16.9,0.0,67.0,14.9
1,AK,2017,28.4,1.6,33.7,204.8,17.6,0.0,81.4,14.6
2,AK,2018,34.0,1.7,34.5,170.6,17.7,0.0,86.3,15.2
3,AK,2019,38.7,1.7,34.3,170.3,17.6,0.0,79.0,15.1
4,AK,2020,39.4,1.8,36.1,147.1,19.1,0.0,68.0,15.1
...,...,...,...,...,...,...,...,...,...,...
403,WY,2019,617.4,39.2,59.0,10.6,8.8,0.0,6.8,10.9
404,WY,2020,618.4,37.2,55.5,13.6,8.7,0.0,5.6,9.7
405,WY,2021,593.4,36.1,54.5,48.8,8.3,0.0,11.3,10.1
406,WY,2022,594.0,37.4,56.3,86.2,8.9,0.0,14.2,10.0


Note that we aren't missing any values

In [66]:
df.isna().sum()

seriesId
stateid    0
period     0
CLEIV      0
CLTCE      0
FFTCE      0
NGEIV      0
NNTCE      0
NUEGV      0
PAEIV      0
PMTCE      0
dtype: int64

Now we can save to a file

In [67]:
df.to_csv("../datasets/cleaned/seds_cleaned.csv", index=False)